Программная реализация MediaPipe
"Интерактивное управление роботом с помощью жестов руки" Стадников Д.И., Кулить Г.В.

In [ ]:
!pip install mediapipe opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 33.4 MB/s eta 0:00:00


In [ ]:
import mediapipe as mp
import cv2
import numpy as np
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Javascript, Image
import io
import PIL.Image


mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

grid_size = 5
robot_position = [2, 2]

# функция движения робота
def move_robot(gesture):
    global robot_position


    x, y = robot_position


    if gesture == "UP":
        x -= 1
    elif gesture == "DOWN":
        x += 1
    elif gesture == "LEFT":
        y -= 1
    elif gesture == "RIGHT":
        y += 1


    # ограничения по сетке
    x = max(0, min(x, grid_size - 1))
    y = max(0, min(y, grid_size - 1))
    robot_position = (x, y)



# построение 2д сетки
def display_grid():
    grid = np.zeros((grid_size, grid_size))
    grid[robot_position[0], robot_position[1]] = 1


    plt.imshow(grid, cmap='cool', interpolation='nearest')
    plt.xticks(ticks=range(grid_size), labels=range(grid_size))
    plt.yticks(ticks=range(grid_size), labels=range(grid_size))
    plt.title("Robot on 2D Grid")
    plt.show()


In [ ]:
def recognize_gesture_with_mediapipe(frame):

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS
                )
                landmarks = np.array([(lm.x, lm.y) for lm in hand_landmarks.landmark])


                if landmarks[8, 1] < landmarks[6, 1] and all(landmarks[finger, 1] > landmarks[finger - 2, 1]
                                                              for finger in [12, 16, 20]):
                    return "UP"

                elif all(landmarks[finger, 1] < landmarks[finger - 2, 1] for finger in [8, 12, 16, 20]):
                    return "LEFT"

                elif all(landmarks[finger, 1] > landmarks[finger - 2, 1] for finger in [8, 12, 16, 20]):
                    return "RIGHT"

                elif (
                    landmarks[8, 1] < landmarks[6, 1] and
                    landmarks[12, 1] < landmarks[10, 1] and
                    landmarks[16, 1] > landmarks[14, 1] and
                    landmarks[20, 1] > landmarks[18, 1]
                ):
                    return "DOWN"
                else:
                    return "UNKNOWN"
        return "NO_HAND"



In [ ]:
def js_to_image(js_reply):
  """
  Params:
          js_reply: JavaScript object containing image from webcam
  Returns:
          img: OpenCV BGR image
  """

  image_bytes = b64decode(js_reply.split(',')[1])

  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)

  img = cv2.imdecode(jpg_as_np, flags=1)

  return img

In [ ]:
def take_photo_and_move_robot_with_mediapipe(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to fit the video element.
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for Capture to be clicked.
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)

    try:
        # Capture photo from webcam
        data = eval_js('takePhoto({})'.format(quality))
        img = js_to_image(data)

        # Recognize gesture using MediaPipe
        gesture = recognize_gesture_with_mediapipe(img)

        # Move the robot based on the recognized gesture
        move_robot(gesture)

        # Annotate the image with the gesture
        annotated_img = cv2.putText(
            img, f"Gesture: {gesture}", (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA
        )
        cv2.imwrite(filename, annotated_img)

        # Display the updated grid
        display_grid()

        # Return the annotated image
        display(Image(filename))

    except Exception as err:
        print(f"Error: {err}")


In [ ]:
def get_number_of_moves():
    while True:
        try:
            moves = int(input("Enter the number of moves: "))
            if moves > 0:
                return moves
            else:
                print("Please enter a positive integer.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

In [ ]:
def play_robot_game_with_mediapipe():

    num_moves = get_number_of_moves()

    print(f"Starting game with {num_moves} moves.")
    print("Make a gesture and press 'Capture' for each move.")

    for move in range(1, num_moves + 1):
        print(f"\nMove {move}/{num_moves}:")
        try:
            filename = take_photo_and_move_robot_with_mediapipe(f"gesture_photo_{move}.jpg")
            print(f"Move {move} processed. Gesture saved to {filename}.")
        except Exception as err:
            print(f"Error during move {move}: {err}")
            break


In [ ]:
play_robot_game_with_mediapipe()